# Canonical Chapter 4 Tables

This notebook reads the canonical analysis datasets and produces auditable Chapter 4 tables. Functional acceptance is reported with explicit passed and failed test-case counts rather than a status label alone.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def locate_analysis_dir(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "clean" / "datasetA.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate reports/analysis from the current working directory.")

ANALYSIS_DIR = locate_analysis_dir(Path.cwd().resolve())
CLEAN_DIR = ANALYSIS_DIR / "data" / "clean"
TABLE_DIR = ANALYSIS_DIR / "data" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_DIR, TABLE_DIR

(PosixPath('/Users/luowei/project/ai-architecture-integrity-study/reports/analysis'),
 PosixPath('/Users/luowei/project/ai-architecture-integrity-study/reports/analysis/data/tables'))

## 1. Read the canonical datasets

All five canonical datasets are loaded first so subsequent Chapter 4 tables share one source-of-truth layer. The checkpoint summary below uses Dataset A.

In [2]:
dataset_a = pd.read_csv(CLEAN_DIR / "datasetA.csv")
dataset_b = pd.read_csv(CLEAN_DIR / "datasetB.csv")
dataset_c = pd.read_csv(CLEAN_DIR / "datasetC.csv")
dataset_d = pd.read_csv(CLEAN_DIR / "datasetD.csv")
dataset_e = pd.read_csv(CLEAN_DIR / "datasetE.csv")

datasets = {
    "Dataset A — Checkpoint Summary": dataset_a,
    "Dataset B — Concern × Checkpoint": dataset_b,
    "Dataset C — Metric Observations": dataset_c,
    "Dataset D — File Findings": dataset_d,
    "Dataset E — T4 Self-Assessment + Efficiency": dataset_e,
}
dataset_manifest = pd.DataFrame([
    {"dataset": name, "rows": len(frame), "columns": len(frame.columns)}
    for name, frame in datasets.items()
])
display(dataset_manifest)
display(dataset_a.head(3))

,dataset,rows,columns
0,Dataset A — Checkpoint Summary,12,19
1,Dataset B — Concern × Checkpoint,228,16
2,Dataset C — Metric Observations,228,26
3,Dataset D — File Findings,58,13
4,Dataset E — T4 Self-Assessment + Efficiency,4,18


,evaluation_id,condition,session_id,agent,strategy,task,task_order,execution_status,functional_acceptance,functional_test_cases_total,functional_test_cases_passed,functional_test_cases_failed,absolute_finding_count,task_local_introduced_findings,task_local_resolved_findings,task_local_net_change,baseline_relative_introduced_findings,baseline_relative_resolved_findings,baseline_relative_net_change
0,session_20260817_130253/T1,Claude Minimal,session_20260817_130253,claude,minimal,T1,1,completed,pass,16,16,0,16,11,0,11,11,0,11
1,session_20260817_130253/T2,Claude Minimal,session_20260817_130253,claude,minimal,T2,2,completed,pass,33,33,0,16,9,9,0,12,1,11
2,session_20260817_130253/T3,Claude Minimal,session_20260817_130253,claude,minimal,T3,3,completed,fail,69,68,1,13,2,5,-3,9,1,8


## 2. Checkpoint Summary with Functional Acceptance

Grain: one shared baseline row followed by one row per `agent × prompt condition × task` checkpoint.

Functional Acceptance is formatted as:

`Overall status — n passed, n failed (n total)`

The architecture columns use the canonical terminology established in Dataset A: Task-Local Introduced Findings, Task-Local Resolved Findings, and Baseline-Relative Introduced Findings.

In [3]:
required_columns = {
    "agent", "strategy", "task", "task_order", "functional_acceptance",
    "functional_test_cases_total", "functional_test_cases_passed",
    "functional_test_cases_failed", "task_local_introduced_findings",
    "task_local_resolved_findings", "baseline_relative_introduced_findings",
}
missing_columns = required_columns.difference(dataset_a.columns)
assert not missing_columns, f"Dataset A is missing required columns: {sorted(missing_columns)}"

acceptance_count_columns = [
    "functional_test_cases_total",
    "functional_test_cases_passed",
    "functional_test_cases_failed",
]
assert dataset_a[acceptance_count_columns].notna().all().all()
assert dataset_a["functional_test_cases_total"].eq(
    dataset_a["functional_test_cases_passed"] + dataset_a["functional_test_cases_failed"]
).all()
assert dataset_a["functional_acceptance"].eq("pass").eq(
    dataset_a["functional_test_cases_failed"].eq(0)
).all()

acceptance_audit = dataset_a[[
    "condition", "task", "functional_acceptance",
    "functional_test_cases_passed", "functional_test_cases_failed",
    "functional_test_cases_total",
]].copy()
display(acceptance_audit)

,condition,task,functional_acceptance,functional_test_cases_passed,functional_test_cases_failed,functional_test_cases_total
0,Claude Minimal,T1,pass,16,0,16
1,Claude Minimal,T2,pass,33,0,33
2,Claude Minimal,T3,fail,68,1,69
3,Claude Structured,T1,pass,16,0,16
4,Claude Structured,T2,fail,7,26,33
5,Claude Structured,T3,fail,6,63,69
6,Codex Minimal,T1,pass,16,0,16
7,Codex Minimal,T2,fail,7,26,33
8,Codex Minimal,T3,fail,9,60,69
9,Codex Structured,T1,fail,15,1,16


In [4]:
CONDITION_ORDER = [
    ("claude", "minimal"),
    ("claude", "structured"),
    ("codex", "minimal"),
    ("codex", "structured"),
]
condition_rank = {condition: rank for rank, condition in enumerate(CONDITION_ORDER)}

agent_rows = dataset_a.copy()
agent_rows["condition_rank"] = [
    condition_rank[(agent, strategy)]
    for agent, strategy in zip(agent_rows["agent"], agent_rows["strategy"])
]
agent_rows = agent_rows.sort_values(["condition_rank", "task_order"]).reset_index(drop=True)
agent_rows["Functional Acceptance"] = agent_rows.apply(
    lambda row: (
        f"{row['functional_acceptance'].title()} — "
        f"{int(row['functional_test_cases_passed'])} passed, "
        f"{int(row['functional_test_cases_failed'])} failed "
        f"({int(row['functional_test_cases_total'])} total)"
    ),
    axis=1,
)

presentation_rows = pd.DataFrame({
    "Agent": agent_rows["agent"].str.title(),
    "Prompt Condition": agent_rows["strategy"].str.title(),
    "Task": agent_rows["task"],
    "Evaluation Checkpoint": "E" + agent_rows["task_order"].astype(int).astype(str),
    "Functional Acceptance": agent_rows["Functional Acceptance"],
    "Task-Local Introduced Findings": agent_rows["task_local_introduced_findings"].astype(int).astype(str),
    "Task-Local Resolved Findings": agent_rows["task_local_resolved_findings"].astype(int).astype(str),
    "Baseline-Relative Introduced Findings": agent_rows["baseline_relative_introduced_findings"].astype(int).astype(str),
})
baseline_row = pd.DataFrame([{
    "Agent": "Shared baseline",
    "Prompt Condition": "—",
    "Task": "—",
    "Evaluation Checkpoint": "E0",
    "Functional Acceptance": "N/A",
    "Task-Local Introduced Findings": "N/A",
    "Task-Local Resolved Findings": "N/A",
    "Baseline-Relative Introduced Findings": "0",
}])
checkpoint_summary_table = pd.concat([baseline_row, presentation_rows], ignore_index=True)

assert len(checkpoint_summary_table) == 13
assert checkpoint_summary_table.iloc[0]["Evaluation Checkpoint"] == "E0"
assert checkpoint_summary_table["Evaluation Checkpoint"].value_counts().to_dict() == {
    "E1": 4, "E2": 4, "E3": 4, "E0": 1,
}
checkpoint_summary_table

,Agent,Prompt Condition,Task,Evaluation Checkpoint,Functional Acceptance,Task-Local Introduced Findings,Task-Local Resolved Findings,Baseline-Relative Introduced Findings
0,Shared baseline,—,—,E0,N/A,N/A,N/A,0
1,Claude,Minimal,T1,E1,"Pass — 16 passed, 0 failed (16 total)",11,0,11
2,Claude,Minimal,T2,E2,"Pass — 33 passed, 0 failed (33 total)",9,9,12
3,Claude,Minimal,T3,E3,"Fail — 68 passed, 1 failed (69 total)",2,5,9
4,Claude,Structured,T1,E1,"Pass — 16 passed, 0 failed (16 total)",9,0,9
5,Claude,Structured,T2,E2,"Fail — 7 passed, 26 failed (33 total)",8,8,10
6,Claude,Structured,T3,E3,"Fail — 6 passed, 63 failed (69 total)",4,5,9
7,Codex,Minimal,T1,E1,"Pass — 16 passed, 0 failed (16 total)",12,0,12
8,Codex,Minimal,T2,E2,"Fail — 7 passed, 26 failed (33 total)",31,9,35
9,Codex,Minimal,T3,E3,"Fail — 9 passed, 60 failed (69 total)",15,16,34


In [5]:
table_style = (
    checkpoint_summary_table.style
    .hide(axis="index")
    .set_properties(**{"text-align": "center", "white-space": "nowrap"})
    .set_table_styles([
        {"selector": "th", "props": [("background-color", "#D9EAF7"), ("font-weight", "bold"), ("text-align", "center")]},
        {"selector": "tbody tr:nth-child(even)", "props": [("background-color", "#F6F8FA")]},
        {"selector": "td", "props": [("padding", "6px 10px"), ("border", "1px solid white")]},
    ])
)
display(table_style)

Agent,Prompt Condition,Task,Evaluation Checkpoint,Functional Acceptance,Task-Local Introduced Findings,Task-Local Resolved Findings,Baseline-Relative Introduced Findings
Shared baseline,—,—,E0,N/A,N/A,N/A,0
Claude,Minimal,T1,E1,"Pass — 16 passed, 0 failed (16 total)",11,0,11
Claude,Minimal,T2,E2,"Pass — 33 passed, 0 failed (33 total)",9,9,12
Claude,Minimal,T3,E3,"Fail — 68 passed, 1 failed (69 total)",2,5,9
Claude,Structured,T1,E1,"Pass — 16 passed, 0 failed (16 total)",9,0,9
Claude,Structured,T2,E2,"Fail — 7 passed, 26 failed (33 total)",8,8,10
Claude,Structured,T3,E3,"Fail — 6 passed, 63 failed (69 total)",4,5,9
Codex,Minimal,T1,E1,"Pass — 16 passed, 0 failed (16 total)",12,0,12
Codex,Minimal,T2,E2,"Fail — 7 passed, 26 failed (33 total)",31,9,35
Codex,Minimal,T3,E3,"Fail — 9 passed, 60 failed (69 total)",15,16,34


## 3. Save and read back

The presentation table is saved as a standalone CSV and read back to verify its row order and displayed values.

In [6]:
table_path = TABLE_DIR / "checkpoint_summary.csv"
checkpoint_summary_table.to_csv(table_path, index=False)
checkpoint_summary_check = pd.read_csv(table_path, dtype=str, keep_default_na=False)
assert checkpoint_summary_check.equals(checkpoint_summary_table)
display(checkpoint_summary_check)
print(f"Saved and validated {len(checkpoint_summary_check)} rows -> {table_path.relative_to(ANALYSIS_DIR)}")

,Agent,Prompt Condition,Task,Evaluation Checkpoint,Functional Acceptance,Task-Local Introduced Findings,Task-Local Resolved Findings,Baseline-Relative Introduced Findings
0,Shared baseline,—,—,E0,N/A,N/A,N/A,0
1,Claude,Minimal,T1,E1,"Pass — 16 passed, 0 failed (16 total)",11,0,11
2,Claude,Minimal,T2,E2,"Pass — 33 passed, 0 failed (33 total)",9,9,12
3,Claude,Minimal,T3,E3,"Fail — 68 passed, 1 failed (69 total)",2,5,9
4,Claude,Structured,T1,E1,"Pass — 16 passed, 0 failed (16 total)",9,0,9
5,Claude,Structured,T2,E2,"Fail — 7 passed, 26 failed (33 total)",8,8,10
6,Claude,Structured,T3,E3,"Fail — 6 passed, 63 failed (69 total)",4,5,9
7,Codex,Minimal,T1,E1,"Pass — 16 passed, 0 failed (16 total)",12,0,12
8,Codex,Minimal,T2,E2,"Fail — 7 passed, 26 failed (33 total)",31,9,35
9,Codex,Minimal,T3,E3,"Fail — 9 passed, 60 failed (69 total)",15,16,34


Saved and validated 13 rows -> data/tables/checkpoint_summary.csv


## 4. Silent Decay Cases with Passing Constraints

This Dataset C table reports the BE-SIZE observations classified as Silent Decay. The metric breaches its upper Tukey fence while the corresponding hard architectural constraint still passes.

In [7]:
required_silent_decay_columns = {
    "agent", "strategy", "task", "task_order", "concern", "metric_id",
    "raw_value", "unit", "direction", "upper_fence",
    "metric_bad_outlier", "constraint_failure", "silent_decay",
}
missing_silent_decay_columns = required_silent_decay_columns.difference(dataset_c.columns)
assert not missing_silent_decay_columns, (
    f"Dataset C is missing required columns: {sorted(missing_silent_decay_columns)}"
)

silent_decay_rows = dataset_c.loc[
    dataset_c["silent_decay"]
    & dataset_c["concern"].eq("BE-SIZE")
    & dataset_c["metric_id"].str.startswith("BE-SIZE-M-001")
].copy()
silent_decay_rows["condition_rank"] = [
    condition_rank[(agent, strategy)]
    for agent, strategy in zip(silent_decay_rows["agent"], silent_decay_rows["strategy"])
]
silent_decay_rows = silent_decay_rows.sort_values(
    ["condition_rank", "task_order"]
).reset_index(drop=True)

assert len(silent_decay_rows) == 2
assert silent_decay_rows["unit"].eq("ratio").all()
assert silent_decay_rows["direction"].eq("lower_is_better").all()
assert silent_decay_rows["metric_bad_outlier"].all()
assert (~silent_decay_rows["constraint_failure"]).all()
assert silent_decay_rows["raw_value"].gt(silent_decay_rows["upper_fence"]).all()

metric_codes = silent_decay_rows["metric_id"].str.extract(r"^(.*?-M-\d+)", expand=False)
silent_decay_table = pd.DataFrame({
    "Agent": silent_decay_rows["agent"].str.title(),
    "Prompt": silent_decay_rows["strategy"].str.title(),
    "Task": silent_decay_rows["task"],
    "Checkpoint": "E" + silent_decay_rows["task_order"].astype(int).astype(str),
    "Concern / Metric": silent_decay_rows["concern"] + " / " + metric_codes,
    r"Observed Metric $M$": silent_decay_rows["raw_value"].map(lambda value: f"{value:.1%}"),
    r"Upper Tukey Fence $U_j$": silent_decay_rows["upper_fence"].map(lambda value: f"{value:.1%}"),
    "Corresponding Constraint Status": silent_decay_rows["constraint_failure"].map(
        {False: "Pass", True: "Fail"}
    ),
})

assert silent_decay_table.to_dict(orient="records") == [
    {
        "Agent": "Claude", "Prompt": "Minimal", "Task": "T2",
        "Checkpoint": "E2", "Concern / Metric": "BE-SIZE / BE-SIZE-M-001",
        r"Observed Metric $M$": "2.4%", r"Upper Tukey Fence $U_j$": "0.0%",
        "Corresponding Constraint Status": "Pass",
    },
    {
        "Agent": "Codex", "Prompt": "Minimal", "Task": "T3",
        "Checkpoint": "E3", "Concern / Metric": "BE-SIZE / BE-SIZE-M-001",
        r"Observed Metric $M$": "1.5%", r"Upper Tukey Fence $U_j$": "0.0%",
        "Corresponding Constraint Status": "Pass",
    },
]
silent_decay_table

,Agent,Prompt,Task,Checkpoint,Concern / Metric,Observed Metric $M$,Upper Tukey Fence $U_j$,Corresponding Constraint Status
0,Claude,Minimal,T2,E2,BE-SIZE / BE-SIZE-M-001,2.4%,0.0%,Pass
1,Codex,Minimal,T3,E3,BE-SIZE / BE-SIZE-M-001,1.5%,0.0%,Pass


In [8]:
silent_decay_style = (
    silent_decay_table.style
    .hide(axis="index")
    .set_properties(**{"text-align": "center", "white-space": "nowrap"})
    .set_table_styles([
        {"selector": "th", "props": [("background-color", "#D9EAF7"), ("font-weight", "bold"), ("text-align", "center")]},
        {"selector": "tbody tr:nth-child(even)", "props": [("background-color", "#F6F8FA")]},
        {"selector": "td", "props": [("padding", "6px 10px"), ("border", "1px solid white")]},
    ])
)
display(silent_decay_style)

Agent,Prompt,Task,Checkpoint,Concern / Metric,Observed Metric $M$,Upper Tukey Fence $U_j$,Corresponding Constraint Status
Claude,Minimal,T2,E2,BE-SIZE / BE-SIZE-M-001,2.4%,0.0%,Pass
Codex,Minimal,T3,E3,BE-SIZE / BE-SIZE-M-001,1.5%,0.0%,Pass


In [9]:
silent_decay_table_path = TABLE_DIR / "silent_decay_cases.csv"
silent_decay_table.to_csv(silent_decay_table_path, index=False)
silent_decay_table_check = pd.read_csv(
    silent_decay_table_path, dtype=str, keep_default_na=False
)
assert silent_decay_table_check.equals(silent_decay_table)
display(silent_decay_table_check)
print(
    f"Saved and validated {len(silent_decay_table_check)} rows -> "
    f"{silent_decay_table_path.relative_to(ANALYSIS_DIR)}"
)

,Agent,Prompt,Task,Checkpoint,Concern / Metric,Observed Metric $M$,Upper Tukey Fence $U_j$,Corresponding Constraint Status
0,Claude,Minimal,T2,E2,BE-SIZE / BE-SIZE-M-001,2.4%,0.0%,Pass
1,Codex,Minimal,T3,E3,BE-SIZE / BE-SIZE-M-001,1.5%,0.0%,Pass


Saved and validated 2 rows -> data/tables/silent_decay_cases.csv


## 5. Table 4.4 — Agent Self-Assessment Calibration Against E3 Harness Findings

**Source:** Dataset E  
**Grain:** one row per agent–prompt condition  
**Comparison:** E3 Harness-detected findings versus the corresponding T4 agent self-assessment

Reported-to-Harness Ratio is the number of Agent-Reported Findings divided by Harness-Detected Findings. Under-Reporting is reported as both the count shortfall and its percentage of Harness findings.

In [10]:
required_calibration_columns = {
    "agent", "strategy", "harness_checkpoint", "review_checkpoint",
    "absolute_finding_count", "self_reported_finding_count",
    "self_report_coverage", "location_agreement",
}
missing_calibration_columns = required_calibration_columns.difference(dataset_e.columns)
assert not missing_calibration_columns, (
    f"Dataset E is missing required columns: {sorted(missing_calibration_columns)}"
)

calibration_rows = dataset_e.copy()
calibration_rows["condition_rank"] = [
    condition_rank[(agent, strategy)]
    for agent, strategy in zip(calibration_rows["agent"], calibration_rows["strategy"])
]
calibration_rows = calibration_rows.sort_values("condition_rank").reset_index(drop=True)
calibration_rows["reported_to_harness_ratio"] = (
    calibration_rows["self_reported_finding_count"]
    / calibration_rows["absolute_finding_count"]
)
calibration_rows["under_reporting_count"] = (
    calibration_rows["absolute_finding_count"]
    - calibration_rows["self_reported_finding_count"]
)
calibration_rows["under_reporting_rate"] = (
    calibration_rows["under_reporting_count"]
    / calibration_rows["absolute_finding_count"]
)

assert len(calibration_rows) == 4
assert calibration_rows["harness_checkpoint"].eq("T3").all()
assert calibration_rows["review_checkpoint"].eq("T4").all()
assert calibration_rows["under_reporting_count"].gt(0).all()
assert (
    calibration_rows["reported_to_harness_ratio"]
    - calibration_rows["self_report_coverage"]
).abs().lt(1e-12).all()
assert (
    calibration_rows["under_reporting_rate"]
    - (1 - calibration_rows["reported_to_harness_ratio"])
).abs().lt(1e-12).all()

self_assessment_calibration_table = pd.DataFrame({
    "Condition": (
        calibration_rows["agent"].str.title()
        + " – "
        + calibration_rows["strategy"].str.title()
    ),
    "Harness-Detected Findings": calibration_rows["absolute_finding_count"].astype(int),
    "Agent-Reported Findings": calibration_rows["self_reported_finding_count"].astype(int),
    "Reported-to-Harness Ratio": calibration_rows["reported_to_harness_ratio"].map(
        lambda value: f"{value:.1%}"
    ),
    "Under-Reporting, n (%)": calibration_rows.apply(
        lambda row: (
            f"{int(row['under_reporting_count'])} "
            f"({row['under_reporting_rate']:.1%})"
        ),
        axis=1,
    ),
    "Location Agreement": calibration_rows["location_agreement"].map(
        lambda value: f"{value:.1%}"
    ),
})

expected_calibration_records = [
    {
        "Condition": "Claude – Minimal",
        "Harness-Detected Findings": 13,
        "Agent-Reported Findings": 9,
        "Reported-to-Harness Ratio": "69.2%",
        "Under-Reporting, n (%)": "4 (30.8%)",
        "Location Agreement": "22.2%",
    },
    {
        "Condition": "Claude – Structured",
        "Harness-Detected Findings": 13,
        "Agent-Reported Findings": 5,
        "Reported-to-Harness Ratio": "38.5%",
        "Under-Reporting, n (%)": "8 (61.5%)",
        "Location Agreement": "20.0%",
    },
    {
        "Condition": "Codex – Minimal",
        "Harness-Detected Findings": 38,
        "Agent-Reported Findings": 6,
        "Reported-to-Harness Ratio": "15.8%",
        "Under-Reporting, n (%)": "32 (84.2%)",
        "Location Agreement": "83.3%",
    },
    {
        "Condition": "Codex – Structured",
        "Harness-Detected Findings": 24,
        "Agent-Reported Findings": 5,
        "Reported-to-Harness Ratio": "20.8%",
        "Under-Reporting, n (%)": "19 (79.2%)",
        "Location Agreement": "80.0%",
    },
]
assert self_assessment_calibration_table.to_dict(orient="records") == expected_calibration_records
self_assessment_calibration_table

,Condition,Harness-Detected Findings,Agent-Reported Findings,Reported-to-Harness Ratio,"Under-Reporting, n (%)",Location Agreement
0,Claude – Minimal,13,9,69.2%,4 (30.8%),22.2%
1,Claude – Structured,13,5,38.5%,8 (61.5%),20.0%
2,Codex – Minimal,38,6,15.8%,32 (84.2%),83.3%
3,Codex – Structured,24,5,20.8%,19 (79.2%),80.0%


In [11]:
self_assessment_calibration_style = (
    self_assessment_calibration_table.style
    .hide(axis="index")
    .set_properties(**{"text-align": "center", "white-space": "nowrap"})
    .set_table_styles([
        {"selector": "th", "props": [
            ("background-color", "#D9EAF7"),
            ("font-weight", "bold"),
            ("text-align", "center"),
        ]},
        {"selector": "tbody tr:nth-child(even)", "props": [
            ("background-color", "#F6F8FA"),
        ]},
        {"selector": "td", "props": [
            ("padding", "6px 10px"),
            ("border", "1px solid white"),
        ]},
    ])
)
display(self_assessment_calibration_style)

Condition,Harness-Detected Findings,Agent-Reported Findings,Reported-to-Harness Ratio,"Under-Reporting, n (%)",Location Agreement
Claude – Minimal,13,9,69.2%,4 (30.8%),22.2%
Claude – Structured,13,5,38.5%,8 (61.5%),20.0%
Codex – Minimal,38,6,15.8%,32 (84.2%),83.3%
Codex – Structured,24,5,20.8%,19 (79.2%),80.0%


In [12]:
self_assessment_calibration_path = TABLE_DIR / "agent_self_assessment_calibration.csv"
self_assessment_calibration_table.to_csv(
    self_assessment_calibration_path, index=False
)
self_assessment_calibration_check = pd.read_csv(
    self_assessment_calibration_path, dtype=str, keep_default_na=False
)
expected_calibration_check = self_assessment_calibration_table.astype(str)
assert self_assessment_calibration_check.equals(expected_calibration_check)
display(self_assessment_calibration_check)
print(
    f"Saved and validated {len(self_assessment_calibration_check)} rows -> "
    f"{self_assessment_calibration_path.relative_to(ANALYSIS_DIR)}"
)

,Condition,Harness-Detected Findings,Agent-Reported Findings,Reported-to-Harness Ratio,"Under-Reporting, n (%)",Location Agreement
0,Claude – Minimal,13,9,69.2%,4 (30.8%),22.2%
1,Claude – Structured,13,5,38.5%,8 (61.5%),20.0%
2,Codex – Minimal,38,6,15.8%,32 (84.2%),83.3%
3,Codex – Structured,24,5,20.8%,19 (79.2%),80.0%


Saved and validated 4 rows -> data/tables/agent_self_assessment_calibration.csv
